In [9]:
import datetime
import json
import os
import time
from multiprocessing import Pool, cpu_count

import ipywidgets as widgets
import numpy as np
import pandas as pd
import requests
from IPython.display import HTML, Markdown, display
from requests.auth import HTTPBasicAuth
from smart_services import SmartServices
from suds.client import Client
import time
from Add_In_Queries import *

In [2]:
def display_scrollable_df(df, max_height="50vh", max_width="90vw"):
    style = f"""
    <div style="
        display: flex;
        justify-content: center;
        padding: 20px;
    ">
        <div style="
            overflow: auto;
            max-height: {max_height};
            max-width: {max_width};
            width: 100%;
            border: 1px solid #444;
            padding: 10px;
            background-color: #000;
            color: #eee;
            font-family: 'Arial Narrow', Arial, sans-serif;
            box-sizing: border-box;
        ">
            {df.to_html(classes='table', border=0, index=True)}
        </div>
    </div>
    """
    return HTML(style)

In [3]:
def safe_call_fields(tasks):
    
    func,fund,field,dateTo=tasks
    
    try:
        return dateTo, field,fund,func(fund,field,dateTo,dateTo)
    except Exception as e:
        return {dateTo: str(e), "args": tasks}
    

In [6]:
def display_ex_post_app(list_of_funds,field_to_use):
    date_start = widgets.Text(
        step=1,
        description="Start (YYYY-MM-DD)",
        disabled=False,
        display="flex",
        flex_flow="column",
        align_items="stretch",
        style={"description_width": "auto"},
    )

    end_date = widgets.Text(
        step=1,
        description="End (YYYY-MM-DD)",
        disabled=False,
        display="flex",
        flex_flow="column",
        align_items="stretch",
        style={"description_width": "auto"},
    )

    def get_data_on_click(b):

        global table

        start = date_start.value
        end = end_date.value

        # Create a date range with end of month frequency
        date_range = pd.date_range(start=start, end=end, freq='ME')

        dico_date={}

        for date in date_range:

            dico_date[get_date_to_string(date)]=date

        tasks = [(get_perf_fields,fund,field,get_date_to_string(date))
            for fund in list_of_funds for field in field_to_use for date in date_range 
        ]

        dico_metrics = {}
        funds_metrics={}
        start_time_sec=time.time()
        with Pool(processes=min(5, cpu_count() * 2)) as pool:
            for date, field, fund,data in pool.imap_unordered(safe_call_fields, tasks):
                temp_date=dico_date[date]
                if not isinstance(data, str):
                    try:
                        if temp_date not in dico_metrics:
                            dico_metrics[temp_date] = {}
                        dico_metrics[temp_date][field] = data['portfolioPerformance'][0]
                        funds_metrics[fund]=pd.DataFrame(dico_metrics).T
                    except Exception as e:

                        print(f"Data not found for fund {fund} for field {field} at date {temp_date}")
        
        finish=time.time()
        
        print(finish-start_time_sec)
        
        table=pd.DataFrame()

        for key in funds_metrics:
            temp=funds_metrics[key]
            temp['Fund']=key
            table=pd.concat([table,temp])

        cols = [c for c in table.columns if c != 'Fund'] + ['Fund']
        table=table[cols]

        dropdown1.options = field_to_use
        dropdown2.options = list_of_funds

        def get_excel(b):

            table.to_excel('Ex Post Data.xlsx', index=True)

            print("File Generated")

        bt_excel = widgets.Button(
            description="Get Excel",
            layout=widgets.Layout(
                display="flex",
                justify_content="center",
                align_items="center",
                spacing="10px",
                width="auto",
            ),
        )

        bt_excel.on_click(get_excel)
        with data_output:
            data_output.clear_output()
            display(display_scrollable_df(table))
            # display(display_scrollable_df(get_perf_catalog()))
            display(bt_excel)
            # display(display_scrollable_df(tables['Data']))


    dropdown1 = widgets.Dropdown(description="Fields:", value=None, options=field_to_use)
    dropdown2 = widgets.Dropdown(description="Funds:", value=None, options=list_of_funds)

    data_output = widgets.Output()
    button_data = widgets.Button(description="Get Data")
    button_data.on_click(get_data_on_click)

    parameters_ui = widgets.VBox(
        [
            widgets.HBox(
                [date_start, end_date, button_data],
                layout=widgets.Layout(
                    display="flex",
                    justify_content="center",
                    align_items="center",
                    spacing="auto",
                    width="auto",
                ),
            ),
            data_output,
        ]
    )

    data = []


    def on_add_constraint_clicked(b):
        row = {"Field": dropdown1.value, "Fund": dropdown2.value}
        data.append(row)
        with constraint_output:
            constraint_output.clear_output()
            display(pd.DataFrame(data))


    add_constraint_btn = widgets.Button(description="Add Filter", button_style="success")
    add_constraint_btn.on_click(on_add_constraint_clicked)

    constraint_output = widgets.Output()
    output = widgets.Output()


    def on_clear_clicked(b):
        data.clear()
        res.clear()
        with constraint_output:
            constraint_output.clear_output()
            display(pd.DataFrame(columns=["Field", "Fund"]))

        with output:
            output.clear_output()


    clear_btn = widgets.Button(description="Clear All", button_style="danger")
    clear_btn.on_click(on_clear_clicked)

    res = {}


    def on_optimize_clicked(b):

        filter_dataframe = pd.DataFrame(data)
        unique_list_funds = set(filter_dataframe["Fund"])
        dico_filter = {}
        for fund in unique_list_funds:
            temp = filter_dataframe[filter_dataframe["Fund"] == fund]
            dico_filter[fund] = list(set(temp["Field"]))

        for key in dico_filter:

            temp = table[table["Fund"] == key][dico_filter[key]]
            res[key] = temp

        with output:
            output.clear_output()
            for key in res:
                display(Markdown("### " + str(key)))
                display(display_scrollable_df(res[key]))


    optimize_btn = widgets.Button(description="Filter", button_style="primary")
    optimize_btn.on_click(on_optimize_clicked)

    constraint_ui = widgets.VBox(
    [
        widgets.VBox([dropdown1, dropdown2]),
        widgets.HBox([add_constraint_btn, clear_btn, optimize_btn]),
        constraint_output,
        output,
    ]
        )

    tab_contents = ["Control", "Analysis"]

    children = [parameters_ui, constraint_ui]
    tab = widgets.Tab()
    tab.children = children
    for i, title in enumerate(tab_contents):
        tab.set_title(i, title)

    display(tab)

In [7]:
fields=pd.read_excel("fields.xlsx")
funds=pd.read_excel('Scope.xlsx',index_col=0)

list_of_funds=list(funds.index)
field_to_use=list(fields['Code'])


In [12]:
display_ex_post_app(list_of_funds,field_to_use)